# Exercise solutions — GANs and DCGAN

**DeepVision Masterclass · Solutions companion**

These are worked reference solutions for the exercises in the matching numbered lesson. Run the parent lesson first when variables or datasets are reused. Dataset-specific experiments are left as clearly marked tasks; no performance scores are fabricated.

## Exercise 1 — Latent dimension ablation
The generator/discriminator layers are defined for 100-dimensional noise. Make the input size configurable and compare sample diversity and quality, not only losses.

In [ ]:
latent_dims=[64,100,128]
# Update Generator's Linear(latent_dim, ...) and all torch.randn(batch,latent_dim).
# Keep training steps, dataset, optimizer and random seeds controlled.

## Exercise 2 — One-sided label smoothing
Use 0.9 for real labels and 0.0 for fake labels. Avoid smoothing fake labels upward.

In [ ]:
real_targets=torch.full((batch_size,1),.9,device=device)
fake_targets=torch.zeros((batch_size,1),device=device)
# real_loss=criterion(D(real),real_targets)
# fake_loss=criterion(D(fake),fake_targets)

## Exercise 3 — Spectral normalization
Apply spectral normalization to discriminator convolution and linear layers.

In [ ]:
from torch.nn.utils import spectral_norm
class SpectralDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(spectral_norm(nn.Conv2d(1,64,4,2,1)),nn.LeakyReLU(.2),spectral_norm(nn.Conv2d(64,128,4,2,1)),nn.BatchNorm2d(128),nn.LeakyReLU(.2),nn.Flatten(),spectral_norm(nn.Linear(128*7*7,1)))
    def forward(self,x): return self.net(x)

## Exercise 4 — Keras custom GradientTape training step

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
zdim=100
Gk=keras.Sequential([layers.Input((zdim,)),layers.Dense(7*7*128),layers.BatchNormalization(),layers.ReLU(),layers.Reshape((7,7,128)),layers.Conv2DTranspose(64,4,strides=2,padding='same'),layers.BatchNormalization(),layers.ReLU(),layers.Conv2DTranspose(1,4,strides=2,padding='same',activation='tanh')])
Dk=keras.Sequential([layers.Input((28,28,1)),layers.Conv2D(64,4,strides=2,padding='same'),layers.LeakyReLU(.2),layers.Conv2D(128,4,strides=2,padding='same'),layers.BatchNormalization(),layers.LeakyReLU(.2),layers.Flatten(),layers.Dense(1)])
bce=keras.losses.BinaryCrossentropy(from_logits=True)
opt_g=keras.optimizers.Adam(2e-4,beta_1=.5); opt_d=keras.optimizers.Adam(2e-4,beta_1=.5)
@tf.function
def train_step(real):
    n=tf.shape(real)[0]; noise=tf.random.normal((n,zdim))
    with tf.GradientTape() as dt, tf.GradientTape() as gt:
        fake=Gk(noise,training=True)
        real_logits=Dk(real,training=True); fake_logits=Dk(tf.stop_gradient(fake),training=True)
        dloss=bce(tf.ones_like(real_logits),real_logits)+bce(tf.zeros_like(fake_logits),fake_logits)
        gen_logits=Dk(fake,training=True); gloss=bce(tf.ones_like(gen_logits),gen_logits)
    gd=gt.gradient(gloss,Gk.trainable_variables); dd=dt.gradient(dloss,Dk.trainable_variables)
    opt_g.apply_gradients(zip(gd,Gk.trainable_variables)); opt_d.apply_gradients(zip(dd,Dk.trainable_variables))
    return dloss,gloss
# Feed MNIST images normalized to [-1,1] and shaped (batch,28,28,1).

## Exercise 5 — Mode collapse and FID
Mode collapse means the generator produces limited varieties. Inspect fixed-noise samples over epochs and measure diversity. FID needs a suitable feature extractor and enough samples; MNIST FID is not directly comparable to standard ImageNet FID.

## Reporting template
Record the dataset/source and license, split strategy, seed(s), preprocessing, model configuration, optimizer, epochs, best validation checkpoint, final test metrics, hardware and runtime. Replace illustrative code comments with measured results from your own run.